# GridNet-HD — Fusion on the validation zones: ImageVote (Phase 1)

ต้องมีก่อน: `GridNet/cache/<zone>/points.npz` (LiDAR cache) และ `GridNet/cache2d/<zone>/*.npz` (2D cache จาก colab_train_upernet cell 8)

ทำอะไร: GPU association (projection → z-buffer visibility τ 0.5 m → 8 views ใกล้สุดต่อจุด) → เฉลี่ย class probabilities จาก 2D cache ต่อจุด → mIoU บน val zones
ผลลัพธ์ไป Drive `GridNet/fusion/imagevote/`: `eval.json`, ต่อโซน `imagevote_pred.npy`, `img_prob.npy` (fp16, ใช้ต่อใน late fusion), `assoc.npz` (association table ใช้ซ้ำได้)

Runtime: L4 พอ (association ~2–5 นาที/โซน) · A100 เร็วกว่า · RAM: ใช้ High-RAM ถ้ามีให้เลือก (t4z1 80 M จุด)

In [ ]:
#@title 1. GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "no GPU runtime"


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
CACHE_DRIVE = f'{DRIVE}/cache'
RUN2D = 'upernet_convnext_t_v2'                  #@param {type:"string"}
CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN2D}'       # 2D cache of that UPerNet run (v1 legacy path: cache2d/<zone>)
OUT = f'{DRIVE}/fusion/imagevote_{RUN2D}'
ZONES = 't1z5b,t1z8,t3z4,t4z1,t5b1,t5b5'         #@param {type:"string"}
K = 8                   #@param {type:"integer"}
TAU = 0.5               #@param {type:"number"}
CHUNK = 20000000        #@param {type:"integer"}
SAVE_ASSOC = True       #@param {type:"boolean"}
COPY_CACHE2D_LOCAL = False  #@param {type:"boolean"}
os.makedirs(OUT, exist_ok=True)
zones = [z.strip() for z in ZONES.split(',') if z.strip()]
print("zones:", zones, "-> results in", OUT)


In [ ]:
#@title 3. Environment (restore tarball from Drive)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    assert os.path.exists(TAR), "no env tarball on Drive: run colab_train_litept cell 3 once"
    subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
PY = f'{ENV}/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pillow'], check=True)
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/code'); import gridnet.train.assoc_gpu, gridnet.train.imagevote; print('imports ok')"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])


In [ ]:
#@title 5. Data: LiDAR caches -> local disk, camera pose files from Hugging Face, (optional) 2D cache -> local disk
import os, shutil, subprocess, time
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True); os.makedirs('/content/cache', exist_ok=True)
t0 = time.time()
for z in zones:
    dst = f'/content/cache/{z}/points.npz'
    if not os.path.exists(dst):
        os.makedirs(f'/content/cache/{z}', exist_ok=True); shutil.copy(f'{CACHE_DRIVE}/{z}/points.npz', dst)
print(f"LiDAR caches ready in {(time.time() - t0) / 60:.1f} min")
code = ("import sys; sys.path.insert(0,'/content/code'); from gridnet.train.data2d import fetch_zones; "
        f"fetch_zones({zones!r}, '{DATA}', what=('pose',))")
subprocess.run([PY, '-c', code], check=True)
print("pose files:", {z: sorted(os.listdir(f'{DATA}/{z}/pose')) for z in zones})
CACHE2D = CACHE2D_DRIVE
if COPY_CACHE2D_LOCAL:
    CACHE2D = '/content/cache2d'; t0 = time.time()
    for z in zones:
        shutil.copytree(f'{CACHE2D_DRIVE}/{z}', f'{CACHE2D}/{z}', dirs_exist_ok=True)
    print(f"2D cache copied in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 6. ImageVote on the validation zones
import subprocess, time, os
cmd = [PY, '/content/code/scripts/eval_imagevote.py', '--cache', '/content/cache', '--data', DATA, '--cache2d', CACHE2D,
       '--zones', ','.join(zones), '--out', OUT, '--k', str(K), '--tau', str(TAU), '--chunk', str(CHUNK)]
if SAVE_ASSOC: cmd.append('--save-assoc')
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore')
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
try:
    for line in p.stdout:
        if 'Warning' not in line: print(line, end='')
    p.wait(); print(f"exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
except KeyboardInterrupt:
    p.terminate(); p.wait(); print("stopped")


In [ ]:
#@title 7. Figure: confusion matrix + mIoU per zone (ImageVote)
import subprocess, sys, os
from IPython.display import Image, display
fig = f'{OUT}/imagevote_val.png'
r = subprocess.run([sys.executable, '/content/code/scripts/plot_eval.py', '--eval', f'{OUT}/eval.json', '--out', fig,
                    '--title', 'ImageVote (UPerNet ConvNeXt-T, 8 nearest views) — validation zones'],
                   capture_output=True, text=True, env=dict(os.environ, PYTHONPATH='/content/code'))
print(r.stdout.strip(), r.stderr[-500:])
display(Image(fig)) if os.path.exists(fig) else None


## หมายเหตุ
- จุดที่ไม่มี view ที่มองเห็นเลย (ปกติ < 1–2 %) ถูกทายเป็น herbaceous; `eval.json` รายงานทั้ง mIoU รวม และ mIoU เฉพาะจุดที่มี view
- `assoc.npz` ต่อโซน ~1 GB ต่อ 8 ล้านจุด (t4z1 ≈ 10 GB) — ปิด SAVE_ASSOC ได้ถ้าไม่ต้องการ
- จบแล้ว **Disconnect and delete runtime**